In [3]:
import pandas as pd
df = pd.read_excel('basket_data.xlsx')
df

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,date,customer_id,item_name
0,2000-01-01,1,yogurt
1,2000-01-01,1,pork
2,2000-01-01,1,sandwich bags
3,2000-01-01,1,lunch meat
4,2000-01-01,1,all- purpose
...,...,...,...
22338,2002-02-26,1139,soda
22339,2002-02-26,1139,laundry detergent
22340,2002-02-26,1139,vegetables
22341,2002-02-26,1139,shampoo


In [4]:
dataset = df.groupby('customer_id')['item_name'].apply(list)
dataset

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,item_name
customer_id,
1,"[yogurt, pork, sandwich bags, lunch meat, all-..."
2,"[toilet paper, shampoo, hand soap, waffles, ve..."
3,"[soda, pork, soap, ice cream, toilet paper, di..."
4,"[cereals, juice, lunch meat, soda, toilet pape..."
5,"[sandwich loaves, pasta, tortillas, mixes, han..."
...,...
1135,"[sugar, beef, sandwich bags, hand soap, paper ..."
1136,"[coffee/tea, dinner rolls, lunch meat, spaghet..."
1137,"[beef, lunch meat, eggs, poultry, vegetables, ..."


教師なし学習の連関分析を行うために「mlxtend」というライブラリをインストールする

In [1]:
!pip install mlxtend==0.23.1

複数の顧客によって購入されている頻出商品の確認

In [5]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori

te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df2 = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = apriori(df2, min_support=0.04, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False).reset_index(drop=True)
frequent_itemsets

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,support,itemsets
0,0.739245,(vegetables)
1,0.421422,(poultry)
2,0.398595,(ice cream)
3,0.395961,(cereals)
4,0.395083,(lunch meat)
...,...,...
19600,0.040386,"(cheeses, poultry, cereals, dinner rolls)"
19601,0.040386,"(cheeses, mixes, cereals, dinner rolls)"
19602,0.040386,"(ice cream, waffles, spaghetti sauce, lunch meat)"
19603,0.040386,"(ice cream, waffles, poultry, lunch meat)"


「vegetables」や「poultry」は人気の高い商品であることが確認できる

次に連関分析を行う

In [6]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(frequent_itemsets, metric = "lift", min_threshold = 1)
rules = rules.sort_values('support', ascending = False).reset_index(drop=True)

rules

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(poultry),(vegetables),0.421422,0.739245,0.331870,0.787500,1.065276,0.020336,1.227083,0.105908
1,(vegetables),(poultry),0.739245,0.421422,0.331870,0.448931,1.065276,0.020336,1.049919,0.234995
2,(vegetables),(eggs),0.739245,0.389816,0.326602,0.441805,1.133370,0.038433,1.093139,0.451287
3,(eggs),(vegetables),0.389816,0.739245,0.326602,0.837838,1.133370,0.038433,1.607989,0.192852
4,(yogurt),(vegetables),0.384548,0.739245,0.319579,0.831050,1.124188,0.035304,1.543388,0.179492
...,...,...,...,...,...,...,...,...,...,...
197939,(fruits),"(beef, cheeses, ice cream)",0.370500,0.076383,0.040386,0.109005,1.427085,0.012086,1.036613,0.475411
197940,(beef),"(fruits, cheeses, ice cream)",0.374890,0.083406,0.040386,0.107728,1.291606,0.009118,1.027258,0.361169
197941,(cheeses),"(fruits, beef, ice cream)",0.390694,0.080773,0.040386,0.103371,1.279775,0.008829,1.025203,0.358790
197942,(ice cream),"(fruits, beef, cheeses)",0.398595,0.075505,0.040386,0.101322,1.341922,0.010290,1.028727,0.423675


vegetablesを買うとpoultryを買う人の組み合わせや、vegetablesを買うととeggを買う人の組み合わせが多いことが分かる。